# 📱 Notebook 7: Chat Interface
### Talk to your MBA knowledge base from your phone

---

## Two Ways to Chat on Mobile — Both Free

| Option | What it is | Best for |
|--------|-----------|----------|
| **Telegram Bot** | Native chat app on your phone | Quick questions on the go |
| **Streamlit Web UI** | Chat interface in mobile browser | When you want a clean visual UI |

### What about WhatsApp?
WhatsApp requires **Meta Business verification** (not available for personal use) or **Twilio** (~$0.005/message, paid). Telegram is the practical free alternative — same experience, completely free.

---

## How This Works

```
You on your phone
      ↓  (type a question)
Telegram / Streamlit
      ↓  (sends to Python running in Colab)
RAG Pipeline (same as Notebook 6)
  → embed question
  → find top 5 chunks
  → ask GPT-4
      ↓
Answer sent back to your phone
```

**Note:** Colab must be running while you use either interface. If Colab disconnects, just re-run the setup cells.

---

**Time needed:** ~20 minutes | **Cost:** $0 (only pays per question as usual)

In [ ]:
#@title ⚙️ Step 1 — Install all required libraries  { display-mode: "form" }
!pip install openai python-telegram-bot streamlit pyngrok -q
print("✅ All libraries installed")

In [ ]:
#@title ⚙️ Step 2 — Connect Drive, load API key, load vector store  { display-mode: "form" }
import os, json, math
import openai

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")

PROJECT_ROOT = '/content/drive/MyDrive/MBA_RAG'
VECTOR_STORE = os.path.join(PROJECT_ROOT, '.tmp/vector_store.json')

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_API_KEY and IN_COLAB:
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
        if OPENAI_API_KEY:
            os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    except Exception:
        pass

if OPENAI_API_KEY:
    print(f"✅ API key loaded. Starts with: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  API key not found. Open 🔑 Secrets → Add OPENAI_API_KEY → re-run.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

if IN_COLAB and os.path.exists(VECTOR_STORE):
    with open(VECTOR_STORE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    subjects = sorted(set(c.get('subject', '') for c in chunks if c.get('subject')))
    print(f"✅ Loaded {len(chunks):,} chunks")
    print(f"   Subjects: {subjects}")
elif IN_COLAB:
    print(f"⚠️  vector_store.json not found. Run Notebooks 2 → 3 → 4 first.")
    chunks, subjects = [], []
else:
    chunks, subjects = [], []

def cosine_similarity(a, b):
    dot  = sum(x * y for x, y in zip(a, b))
    na   = math.sqrt(sum(x * x for x in a))
    nb   = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb) if na and nb else 0.0

def retrieve(query, subject=None, top_k=5):
    qv   = client.embeddings.create(model='text-embedding-3-small', input=query).data[0].embedding
    pool = [c for c in chunks if c.get('subject','').lower() == subject.lower()] if subject else chunks
    if subject and not pool:
        pool = chunks
    scored = sorted(pool, key=lambda c: cosine_similarity(qv, c['embedding']), reverse=True)
    return scored[:top_k]

def ask(question, subject=None):
    results = retrieve(question, subject)
    context = '\n\n'.join(f"[{r['filename']}]\n{r['text']}" for r in results)
    sources = list(dict.fromkeys(r['filename'] for r in results))
    prompt  = f"""Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have information on this in my MBA files."
Do NOT make up information.\n\nCONTEXT:\n{context}\n\nQUESTION: {question}"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {"role": "system", "content": "You are an MBA knowledge assistant. Answer using only the provided context. Be concise and clear."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.1
    )
    return f"{response.choices[0].message.content}\n\n📚 Sources: {', '.join(sources)}"

if IN_COLAB:
    print()
    print("✅ RAG functions ready.")
    print("   Option A → Telegram Bot (run Step 3)")
    print("   Option B → Streamlit Web UI (run Step 4)")

---
## Option A: Telegram Bot 💬
### Native mobile chat — no browser needed

### Step 1: Create your bot (2 minutes, one-time)

1. Open Telegram on your phone
2. Search for **@BotFather** and tap it
3. Send the message: `/newbot`
4. BotFather asks for a **name** → type anything (e.g. `My MBA Assistant`)
5. BotFather asks for a **username** → must end in `bot` (e.g. `myMBA_bot`)
6. BotFather gives you a **token** that looks like: `7123456789:AAFxxxxxxxxxxxxxxxxxxxxxx`
7. Copy that token → paste it in the cell below

### Step 2: Run the cell below, then open Telegram and message your bot

In [ ]:
#@title 💬 Option A — Start Telegram Bot  { display-mode: "form" }
#@markdown **Before running:** Get your bot token from @BotFather on Telegram (`/newbot` → follow steps → copy the token)
BOT_TOKEN = "YOUR_BOT_TOKEN_HERE" #@param {type:"string"}

from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    name = update.effective_user.first_name
    await update.message.reply_text(
        f"👋 Hi {name}! I'm your MBA Knowledge Assistant.\n\n"
        "Ask me anything from your MBA files.\n\n"
        "💡 Tips:\n"
        "• Plain question → searches all subjects\n"
        "• Start with 'Finance:' to filter by subject\n\n"
        "Examples:\n"
        "  What is competitive advantage?\n"
        "  Finance: How is WACC calculated?"
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text     = update.message.text.strip()
    subject  = None
    question = text
    if ':' in text:
        parts    = text.split(':', 1)
        possible = parts[0].strip()
        if possible in subjects:
            subject  = possible
            question = parts[1].strip()
    if subject:
        await update.message.reply_text(f"🔍 Searching in {subject}...")
    else:
        await update.message.reply_text("🔍 Searching your MBA files...")
    try:
        answer = ask(question, subject=subject)
        await update.message.reply_text(answer)
    except Exception as e:
        await update.message.reply_text(f"❌ Error: {e}")

async def subjects_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "📚 Available subjects:\n" +
        '\n'.join(f"  • {s}" for s in subjects) +
        "\n\nUsage: SubjectName: Your question"
    )

bot_app = ApplicationBuilder().token(BOT_TOKEN).build()
bot_app.add_handler(CommandHandler("start",    start))
bot_app.add_handler(CommandHandler("subjects", subjects_cmd))
bot_app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

print("✅ Bot is running!")
print("   Open Telegram → search for your bot → send /start")
print("   Keep this cell running while you chat.")
print("   To stop: click the ■ stop button next to this cell")

bot_app.run_polling()

### How to use the Telegram bot

| You type | What happens |
|----------|--------------|
| `/start` | Welcome message + instructions |
| `/subjects` | Lists all your subject folders |
| `What is competitive advantage?` | Searches all subjects |
| `Finance: How is WACC calculated?` | Searches Finance folder only |
| `Strategy: What frameworks did I study?` | Searches Strategy folder only |

**While Colab is running → bot works on your phone, even with screen off.**

Colab free tier disconnects after ~90 minutes of no browser activity. To keep it alive longer, just scroll through the notebook occasionally — or upgrade to Colab Pro ($10/month) for longer sessions.

---
## Option B: Streamlit Web UI 🌐
### Chat interface in your mobile browser

This creates a clean chat app accessible via a URL you can open on your phone.

How it works:
1. Streamlit runs a web app inside Colab
2. ngrok creates a public URL that tunnels to it
3. You open that URL on your phone — it looks like a chat app
4. Works as long as Colab is running

In [ ]:
#@title 🌐 Option B — Write Streamlit app file  { display-mode: "form" }
import os, json

subjects_str = json.dumps(subjects)

app_code = f'''import streamlit as st
import openai, json, math, os

OPENAI_API_KEY = "{OPENAI_API_KEY}"
VECTOR_STORE   = "{VECTOR_STORE}"
SUBJECTS       = {subjects_str}

@st.cache_resource
def load_chunks():
    with open(VECTOR_STORE, "r", encoding="utf-8") as f:
        return json.load(f)

client = openai.OpenAI(api_key=OPENAI_API_KEY)
chunks = load_chunks()

def cosine_similarity(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na  = math.sqrt(sum(x*x for x in a))
    nb  = math.sqrt(sum(x*x for x in b))
    return dot/(na*nb) if na and nb else 0.0

def retrieve(query, subject=None, top_k=5):
    qv   = client.embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding
    pool = [c for c in chunks if c.get("subject","").lower()==subject.lower()] if subject else chunks
    if subject and not pool: pool = chunks
    scored = sorted(pool, key=lambda c: cosine_similarity(qv, c["embedding"]), reverse=True)
    return scored[:top_k]

def ask(question, subject=None):
    results = retrieve(question, subject)
    context = "\\n\\n".join(f"[{{r['filename']}}]\\n{{r['text']}}" for r in results)
    sources = list(dict.fromkeys(r["filename"] for r in results))
    prompt  = f"Use ONLY the context below to answer. If not found, say so. Do NOT make up information.\\n\\nCONTEXT:\\n{{context}}\\n\\nQUESTION: {{question}}"
    resp    = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {{"role": "system", "content": "You are an MBA knowledge assistant. Use only the provided context."}},
            {{"role": "user",   "content": prompt}}
        ],
        temperature=0.1
    )
    return resp.choices[0].message.content, sources

st.set_page_config(page_title="MBA Assistant", page_icon="🎓", layout="centered")
st.title("🎓 MBA Knowledge Assistant")
st.caption(f"{{len(chunks):,}} chunks loaded from your MBA files")

with st.sidebar:
    st.header("Filter by Subject")
    subject_choice   = st.selectbox("Subject (optional)", ["All subjects"] + SUBJECTS)
    selected_subject = None if subject_choice == "All subjects" else subject_choice
    st.divider()
    st.caption("Available subjects:")
    for s in SUBJECTS:
        st.caption(f"  • {{s}}")

if "messages" not in st.session_state:
    st.session_state.messages = [{{"role": "assistant", "content": "Hi! Ask me anything about your MBA files."}}]

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask about your MBA..."):
    st.session_state.messages.append({{"role": "user", "content": prompt}})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Searching your MBA files..."):
            try:
                answer, sources = ask(prompt, subject=selected_subject)
                st.write(answer)
                st.caption("📚 Sources: " + ", ".join(sources))
                full_response = answer + "\\n\\n📚 Sources: " + ", ".join(sources)
            except Exception as e:
                full_response = f"❌ Error: {{e}}"
                st.error(full_response)
    st.session_state.messages.append({{"role": "assistant", "content": full_response}})
'''

with open('/content/mba_chat.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ Streamlit app saved to /content/mba_chat.py")
print("   Now run Step 4b to launch it.")

In [ ]:
#@title 🌐 Option B — Launch Streamlit (get your public URL)  { display-mode: "form" }
import subprocess, time
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(1)

process = subprocess.Popen(
    ['streamlit', 'run', '/content/mba_chat.py',
     '--server.port=8501', '--server.headless=true', '--server.enableCORS=false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)
tunnel = ngrok.connect(8501)
url    = tunnel.public_url

print("=" * 60)
print("✅ Your MBA chat app is live!")
print()
print(f"   🔗 Open this URL on your phone:")
print(f"   {url}")
print()
print("   Works as long as this Colab session is running.")
print("=" * 60)

---
## Which Should You Use?

| | Telegram Bot | Streamlit |
|--|-------------|----------|
| Opens in | Telegram app (native) | Mobile browser |
| Feels like | WhatsApp chat | ChatGPT web |
| Subject filter | Type `Finance: question` | Dropdown in sidebar |
| See chat history | ✅ Yes (Telegram keeps it) | ✅ Yes (while tab is open) |
| Setup effort | 2 min (BotFather) | Zero extra setup |
| Recommendation | ✅ Best for mobile chat | ✅ Best for visual UI |

**You can run both at the same time** — just run the Telegram cell AND the Streamlit cells. They both use the same RAG pipeline.

---
## ✅ Notebook 7 Complete!

### What you built:
1. **Telegram bot** — ask MBA questions from Telegram on your phone, for free
2. **Streamlit chat UI** — clean web interface accessible from any device
3. **Subject filtering** — both interfaces support filtering by subject folder

### The complete system:
```
Your phone (Telegram / Browser)
        ↓  question
Colab (running Notebook 7)
        ↓  embed → retrieve 5 chunks → GPT-4
vector_store.json (Google Drive)
        ↓  answer + sources
Your phone
```

### Next step: Phase 2
Run all 1,250 files through Notebooks 2 → 3 → 4 to build the full vector store.
Then come back to this notebook — everything works the same, just with 1,250 files instead of 5.

**Update README.md — check off Notebook 7 ✅**